In [ ]:
import numpy as np
import pandas as pd
import nibabel as nb
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.regression.mixed_linear_model as MixedLM
import scipy.io 
from glob import glob
from os.path import join as pjoin
import os
import matplotlib.pyplot as plt

# setup folders and files to look for
base_dir = '/mnt/tambinidata/ScanTrain/'
analysis_dir = base_dir + 'analysis'
data_dir = pjoin(base_dir, 'data')
der_dir = pjoin(data_dir, 'derivatives')
Nses = [17, 18, 18]
ss_list = ['sub-001', 'sub-002', 'sub-003']
Ns = 3

model_name = 'spm_localizer_model5_run'
contrast = '*con*_fractal_seq_vs_fractal_non_seq*'
out_name = 'fractalseq-vs-nonseq'
# contrast = '*con*_fractals_vs_baseline*'
# out_name = 'fractal-vs-baseline'

# load in other misc info - can remove
dates = scipy.io.loadmat(pjoin(analysis_dir, 'ss_date_values.mat'))
dates = dates['date_values']
info = scipy.io.loadmat(pjoin(analysis_dir, 'localizer_scan_info.mat'))
meanfd = info['meanFD']
info = info['scans_keep']

print(dates)

In [ ]:
# can skip all of this (for normalizing files & loading in nuisance variables, yours are already normalized)
save_dir = pjoin(data_dir, 'group', 'localizer_scans_ds3')
ants_dir = pjoin(data_dir, 'anat', 'ants')

ants_template = glob(pjoin(ants_dir, 'ants-template-ds3.nii*'))
assert(len(ants_template)==1)

cmd_base = 'antsApplyTransforms -d 3 -i '
cmd_ref = ' -r ' + ants_template[0] + ' '

cntr=0

exclude = nb.load(ants_template[0]).get_data() < .5
exclude = exclude.astype('int')

X_mot = []
X_date = []
X_sub = []
X_run = []
for iss in np.arange(0,Ns):
    #print(iss)
    ss = ss_list[iss]
    warp_file = glob(pjoin(ants_dir, '*' + ss + '*T1w' + str(iss) + 'Warp.nii*' ))
    assert(len(warp_file)==1)
    affine_file = glob(pjoin(ants_dir, '*' + ss + '*Affine.txt' ))
    assert(len(affine_file)==1)
    
    cmd_t = ' -t ' + warp_file[0] + ' -t ' + affine_file[0]
    
    s_info = info[:,:,iss]
    ses_list = np.where(np.sum(s_info,axis=1)>0)[0]
    #print(ses_list)
    for ises in ses_list: #np.arange(0, Nses[iss]):
        #print(ises)
        ses = ises+1
        if ses < 10:
            pad = '00'
        else:
            pad = '0'
        ses_str =  'ses-' + pad + str(ses)
        
        run_list = np.where(s_info[ises,:])[0]
        #print(ses_str)
        #print(run_list)
        for irun in run_list: #np.arange(0, Nruns):
            run = irun+1
            ses_dir = pjoin(der_dir, ss, ses_str, model_name + str(run))
            con_file = glob(pjoin(ses_dir, contrast))
            assert(len(con_file)==1)
            basename = os.path.basename(con_file[0])
            #print(basename)
            new_file = pjoin(save_dir, 'ants_' + basename[:-4] + '_run' + str(run) + '.nii.gz')
            #
            #print(new_file)
            if not os.path.exists(new_file):
                cmd = cmd_base + con_file[0] + cmd_ref + '-o ' + new_file + cmd_t
                os.system(cmd)
                
                print('here')
            
            temp = nb.load(new_file).get_data()==0    
            #print(np.sum(exclude>0))
            exclude = temp.astype('int') + exclude
            cntr = cntr+1      
            
            X_mot.append(meanfd[ises, irun, iss])
            X_date.append(dates[ises, iss])
            X_sub.append(iss+1)
            X_run.append(run)

In [ ]:
new_file

In [ ]:
# modify to define group-level mask you are using
mask = exclude==0

# cntr will instead be total # of datapoints you are loading in
Y = np.zeros((np.sum(mask), cntr))
cntr=0

# loop over subjects
for iss in np.arange(0,Ns):

    ss = ss_list[iss]
    s_info = info[:,:,iss]
    ses_list = np.where(np.sum(s_info,axis=1)>0)[0]
    #print(ses_list)

    # loop over sessions
    for ises in ses_list: #np.arange(0, Nses[iss]):
        #print(ises)
        ses = ises+1
        if ses < 10:
            pad = '00'
        else:
            pad = '0'
        ses_str =  'ses-' + pad + str(ses)
        
        # loop over runs - you don't need to do this since you are loading in session-level data
        run_list = np.where(s_info[ises,:])[0]
        #print(ses_str)
        #print(run_list)
        for irun in run_list: 
            run = irun+1
            ses_dir = pjoin(der_dir, ss, ses_str, model_name + str(run))

            # load in each contrast file
            con_file = glob(pjoin(ses_dir, contrast))
            assert(len(con_file)==1)
            basename = os.path.basename(con_file[0])
            new_file = pjoin(save_dir, 'ants_' + basename[:-4] + '_run' + str(run) + '.nii.gz')
            
            temp = nb.load(new_file).get_data() # this actually loads in data
            Y[:, cntr] = temp[mask] # just select voxels in mask
            cntr = cntr+1      

In [ ]:
Y.shape
print(save_dir)
np.sum(mask)
#np.array(X_run).shape

In [ ]:
Nvox = Y.shape[0]

# Output file names
tmap_name = pjoin(save_dir, 'Tstat_' + out_name + '_Time_mixedLM.nii.gz')
pmap_name = pjoin(save_dir, 'Pmap1m_' + out_name + '_Time_mixedLM.nii.gz')

print(Nvox)
Tvals = np.zeros((Nvox))
Pvals = np.ones((Nvox))

X_date = np.array(X_date)
X_mot = np.array(X_mot)
X_sub = np.array(X_sub)
X_run = np.array(X_run)
Y_name = 'Y'
X_name = 'Time'
X_name2 = 'Motion'
X_name3 = 'Run'

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
    
for ivox in np.arange(0, Nvox):

    # setup dataframe w/ variables for each voxels. In your case you'll have Y (overnight changes),
    # X-session (indicating first/second sessions), and X-sub (indicating subject ID) 
    d = {'Y': Y[ivox,:], 'Time': X_date, 'Motion': X_mot, 'Run': X_run, 'IDS': X_sub}
    df = pd.DataFrame(d)

    # setup mixed model and fit it
    md = MixedLM.MixedLM.from_formula('{} ~ {} + {} + {}'.format(Y_name,X_name,X_name2,X_name3), \
                                        groups = df['IDS'], data = df)
    mdf = md.fit() 

    # populate these vectors w/ T, P values for each voxel
    Tvals[ivox] = mdf.tvalues.Time
    Pvals[ivox] = mdf.pvalues.Time

    if np.remainder(ivox, 100)==0:
        print(str(ivox) + '/' + str(Nvox) )

In [ ]:
out_name

In [ ]:
print(ivox)
aff = nb.load(ants_template[0]).affine

# put T, P values into 3d matrix instead of 2d
Tmap = np.zeros(mask.shape)
Pmap = np.zeros(mask.shape)

Tmap[mask] = Tvals
Pmap[mask] = 1-Pvals

# save output files
tmap_name = pjoin(save_dir, 'Tstat_' + out_name + '_Time_mixedLM.nii.gz')
pmap_name = pjoin(save_dir, 'Pmap1m_' + out_name + '_Time_mixedLM.nii.gz')

new_img = nb.Nifti1Image(Tmap, aff)
new_img.to_filename(tmap_name)

new_img = nb.Nifti1Image(Pmap, aff)
new_img.to_filename(pmap_name)
#np.sum(exclude>0)
#X_sub
#exclude.astype('int')

#np.where(np.sum(info[:,:,1],axis=1)>0)
#np.where(info[7,:,1])
#info[:,:,0]